In [12]:
import json
import re
import os
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score


In [121]:
# === 1️⃣ Recorder function ===
def recorder(out):
    text = out.lower()
    if re.search(r"\b(?:no|not|n't)\b", text):
        return 0
    else:
        return 1

In [148]:
# === Paths ===
pope_type = "random"
method = "greedy"
base_dir = "../opera_log/pope_eval_results/google/gemma-3n-E4B-it/greedy"

detailed_path = os.path.join(base_dir, f"POPE_type_{pope_type}_{method}_detailed.jsonl")
updated_detailed_path = os.path.join(base_dir, f"POPE_type_{pope_type}_{method}_detailed_fixed.jsonl")
metric_path = os.path.join(base_dir, f"POPE_type_{pope_type}_{method}_metric.jsonl")

In [149]:



# === Step 1: Update predictions ===
pred_list, label_list = [], []
updated_entries = []

with open(detailed_path, "r") as fin:
    for line in fin:
        data = json.loads(line)
        response = data.get("response", "")
        pred = recorder(response)
        data["prediction"] = pred
        updated_entries.append(data)

        # store for metric computation
        if "label" in data:
            label_list.append(data["label"])
            pred_list.append(pred)

# Save updated detailed file
with open(updated_detailed_path, "w") as fout:
    for item in updated_entries:
        json.dump(item, fout)
        fout.write("\n")



In [150]:
# === Step 2: Compute metrics ===
os.makedirs(base_dir, exist_ok=True)

cm = confusion_matrix(label_list, pred_list, labels=[1, 0])
acc = accuracy_score(label_list, pred_list)
report_dict = classification_report(label_list, pred_list, output_dict=True)


# === Step 3: Save metrics ===
with open(metric_path, "w") as f:
    json.dump(
        {
            "ConfusionMatrix": cm.tolist(),
            "Accuracy": acc,
            "Report": report_dict,
        },
        f,
        indent=2,
    )

print(f"✅ Updated detailed file: {updated_detailed_path}")
print(f"✅ Metrics saved to: {metric_path}")
print(f"\n📊 Accuracy: {acc:.4f}")

✅ Updated detailed file: ../opera_log/pope_eval_results/google/gemma-3n-E4B-it/greedy/POPE_type_random_greedy_detailed_fixed.jsonl
✅ Metrics saved to: ../opera_log/pope_eval_results/google/gemma-3n-E4B-it/greedy/POPE_type_random_greedy_metric.jsonl

📊 Accuracy: 0.8454
